# Analysis of Pickle Data (RNA & Protein)

The file `20211022_dict_matrix_20dataset.pkl` contains the core RNA and Protein datasets.
Based on preliminary inspection, this file contains a **List** of length 2, where each element is likely a **Dictionary** mapping cohort names to data matrices.

**Goal:**
1. Load the pickle file.
2. separate the two dictionaries (likely Protein and RNA).
3. Inspect the shape and contents of the matrices within.
4. Prepare them for multivariate analysis.

In [1]:
import os
import pandas as pd
import pickle
import numpy as np

# Define paths
DATA_DIR = r"./data/Collection_A"
pkl_path = os.path.join(DATA_DIR, "20211022_dict_matrix_20dataset.pkl")

print(f"Target File: {pkl_path}")

Target File: ./data/Collection_A\20211022_dict_matrix_20dataset.pkl


## 1. Extracting Data

In [2]:
if os.path.exists(pkl_path):
    print("Loading pickle file (this may take a moment)...")
    with open(pkl_path, 'rb') as f:
        data_list = pickle.load(f)
    
    print(f"Object Type: {type(data_list)}")
    if isinstance(data_list, list):
        print(f"List Length: {len(data_list)}")
        
        # Separate the two elements
        dict_1 = data_list[0]
        dict_2 = data_list[1]
        
        print(f"\n--- Component 1 ({type(dict_1)}) ---")
        if isinstance(dict_1, dict):
            print(f"Keys: {len(dict_1)} cohorts")
            print(f"Sample Keys: {list(dict_1.keys())[:5]}")
            
        print(f"\n--- Component 2 ({type(dict_2)}) ---")
        if isinstance(dict_2, dict):
            print(f"Keys: {len(dict_2)} cohorts")
            print(f"Sample Keys: {list(dict_2.keys())[:5]}")
            
    else:
        print("Unexpected structure: Root object is not a list.")
else:
    print("File not found.")

Loading pickle file (this may take a moment)...
Object Type: <class 'list'>
List Length: 2

--- Component 1 (<class 'dict'>) ---
Keys: 20 cohorts
Sample Keys: ['colon_86_2014_labelfree', 'prostate_65_2019_labelfree', 'lung_76_2020_labelfree', 'liver_62_2019_labelfree', 'brain_71_2017_labelfree']

--- Component 2 (<class 'pandas.core.frame.DataFrame'>) ---


## 2. Investigating Contents

We need to determine which dictionary corresponds to **Protein** and which to **RNA**.
We will check specific dataframe characteristics (shape, values) to infer the type.

In [3]:
def inspect_dictionary(d, name="Dict"):
    print(f"\nEvaluating contents of {name}...")
    first_key = list(d.keys())[0]
    sample_data = d[first_key]
    
    print(f"  Cohort: {first_key}")
    print(f"  Type: {type(sample_data)}")
    
    if hasattr(sample_data, 'shape'):
        print(f"  Shape: {sample_data.shape}")
    
    if isinstance(sample_data, pd.DataFrame):
        print("  Columns (Features) Sample:", list(sample_data.columns)[:5])
        print("  Index (Samples) Sample:", list(sample_data.index)[:5])
        # Check for gene IDs (ENSG...) often found in RNA-seq
        is_rna_likely = any("ENSG" in str(c) for c in sample_data.columns) or any("ENSG" in str(c) for c in sample_data.index)
        print(f"  Likely RNA (ENSG check)? {is_rna_likely}")

if isinstance(data_list, list) and len(data_list) == 2:
    inspect_dictionary(dict_1, "Component 1")
    inspect_dictionary(dict_2, "Component 2")


Evaluating contents of Component 1...
  Cohort: colon_86_2014_labelfree
  Type: <class 'dict'>

Evaluating contents of Component 2...
  Cohort: common
  Type: <class 'pandas.core.series.Series'>
  Shape: (20,)


## 3. Data Consolidation

If these are indeed our cohorts, we can proceed to merge them into a single matrix for analysis, similar to the logic in `process_collection_a.ipynb` but using this pre-compiled source.